## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [8]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [5]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 1024], stddev=tf.sqrt(2.0/784)))
        self.b1 = tf.Variable(tf.zeros([1024]))
        self.W2 = tf.Variable(tf.random.normal([1024, 10], stddev=tf.sqrt(2.0/1024)))
        self.b2 = tf.Variable(tf.zeros([10]))
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, (-1, 784))
        # 使用 Leaky ReLU 激活函数，避免神经元死亡
        h = tf.nn.leaky_relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [6]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.489586 ; accuracy 0.05293333
epoch 1 : loss 2.4540458 ; accuracy 0.06406666
epoch 2 : loss 2.4211917 ; accuracy 0.07503334
epoch 3 : loss 2.3905234 ; accuracy 0.08605
epoch 4 : loss 2.3616643 ; accuracy 0.0982
epoch 5 : loss 2.334325 ; accuracy 0.11043333
epoch 6 : loss 2.3082802 ; accuracy 0.124066666
epoch 7 : loss 2.2833495 ; accuracy 0.13813333
epoch 8 : loss 2.2593884 ; accuracy 0.15306666
epoch 9 : loss 2.236278 ; accuracy 0.1673
epoch 10 : loss 2.213921 ; accuracy 0.18298334
epoch 11 : loss 2.1922357 ; accuracy 0.19891667
epoch 12 : loss 2.1711535 ; accuracy 0.21706666
epoch 13 : loss 2.1506174 ; accuracy 0.23498334
epoch 14 : loss 2.1305783 ; accuracy 0.25258332
epoch 15 : loss 2.1109953 ; accuracy 0.27046666
epoch 16 : loss 2.0918326 ; accuracy 0.28695
epoch 17 : loss 2.0730598 ; accuracy 0.303
epoch 18 : loss 2.0546513 ; accuracy 0.31858334
epoch 19 : loss 2.036584 ; accuracy 0.3338
epoch 20 : loss 2.0188382 ; accuracy 0.34851667
epoch 21 : loss 2.0013976 ; a